# Scoring Risque Credit — Demonstration

> **Auteur :** Emmanuel TSAGUE — Data Scientist / Data Analyst  
> **Données :** Simulées / Anonymisées — Portfolio pédagogique  
> **GitHub :** https://github.com/TSAGUE25


## 1. Imports

Librairies pour le scoring credit et la calibration.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from scipy.stats import ks_2samp
np.random.seed(42)
print('Imports OK')

## 2. Dataset simule — 20 000 dossiers credit

Simulation de dossiers de credit avec probabilite de defaut.


In [ ]:
N = 20000
df = pd.DataFrame({
    'revenu_mensuel':    np.abs(np.random.lognormal(7.5, 0.5, N)),
    'taux_endettement':  np.random.beta(2, 5, N),
    'anciennete_banque': np.random.randint(0, 30, N),
    'incidents_passes':  np.random.poisson(0.3, N),
    'nb_credits':        np.random.randint(0, 8, N),
    'proprietaire':      np.random.randint(0, 2, N),
})
logit = (-2.0 + 0.03*df['taux_endettement']*10 +
          0.8*(df['incidents_passes']>0) - 0.3*df['proprietaire'] +
          np.random.normal(0, 0.5, N))
df['defaut'] = np.random.binomial(1, 1/(1+np.exp(-logit)))
print(f'Taux de defaut: {df.defaut.mean():.1%}')

## 3. Modele + Calibration + KS Statistic

GradientBoosting calibre + KS Statistic standard du secteur bancaire.


In [ ]:
X = df.drop(columns='defaut')
y = df['defaut']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

base_model = Pipeline([('scaler', StandardScaler()),
                        ('gb', GradientBoostingClassifier(n_estimators=200, random_state=42))])
cal_model = CalibratedClassifierCV(base_model, method='isotonic', cv=3)
cal_model.fit(X_train, y_train)

y_proba = cal_model.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, y_proba)
gini = 2*auc - 1

ks_stat, _ = ks_2samp(y_proba[y_test==0], y_proba[y_test==1])

print(f'AUC   : {auc:.4f}')
print(f'Gini  : {gini:.4f}')
print(f'KS    : {ks_stat:.4f} ({"Bon" if ks_stat > 0.4 else "Acceptable"})')